# Final Project - Fast High-Accuracy YOLO Continuation

This notebook is the speed-focused copy of `01B_YOLO_BDD100K_Overnight_Accuracy.ipynb`.
It leaves the executed `01B` notebook and all `v2` checkpoints unchanged. The default
profile automatically starts from the newest accuracy checkpoint, trains at efficient
resolutions, reuses prior class-aware scenes, and uses a fixed 1,200-image
validation subset during epochs. Final model selection still uses the full 8,000-image
validation split and the final report still uses the held-out 2,000-image test split.

## Why this is faster

The measured `v2` run spent 23 minutes reading 70,000 label files and about 56-65 minutes
per 4,000-image epoch at 704 pixels. This workflow:

- keeps all class-aware `v2` scenes and adds fresh scenes without rescanning labels;
- uses 576-pixel main training and 640-pixel refinement;
- uses measured batch sizes of 10 at 576px and 8 at 640px;
- RAM-caches the refinement set and disables deterministic CUDA kernels;
- validates on 1,200 fixed images during epochs, then validates finalists on all 8,000;
- compares the starting, main, and refined checkpoints so extra training cannot silently
  replace a stronger model.

A local fit check processed batch 10 at 576px in 3.0 seconds per step, versus batch 6
at about 5.2 seconds per step in the recorded `v2` run. That is roughly 2.9 times more
training images per second, and the smaller epoch-validation split removes most of the
old validation overhead.

Object detection has no single classification accuracy. The notebook reports precision,
recall, F1, mAP@50, and mAP@50:95. The 70-80% target is measured, never manufactured by
a confidence threshold.

In [ ]:
from pathlib import Path
import json
import math
import random
import shutil
import sys
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display
from ultralytics import YOLO

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from road_detection.yolo_training_utils import (
    latest_best_checkpoint,
    load_dataset_config,
    prepare_fast_data_files,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_float32_matmul_precision("high")

print(f"Project: {PROJECT_ROOT}")
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__} | CUDA build: {torch.version.cuda}")

## Select the continuation profile

`fast_overnight` is tuned for this laptop's 4 GB RTX 3050. Use a new `RUN_TAG` for a new
experiment. If a stage is interrupted, keep the same tag and set that stage's `RESUME`
flag to `True`. `START_WEIGHTS_OVERRIDE` is optional; by default the newest prior
`runs/notebooks/yolo_accuracy/*/weights/best.pt` is selected automatically.

In [ ]:
RUN_MODE = "fast_overnight"  # fast_smoke | fast_overnight | larger_gpu | cpu_fallback
RUN_TAG = "v3_fast"
PREFERRED_MANIFEST_TAG = "v2"

RUN_MAIN_TRAINING = True
RUN_REFINEMENT = True
RESUME_MAIN = False
RESUME_REFINEMENT = False
START_WEIGHTS_OVERRIDE = None  # Example: r"runs/.../weights/best.pt"

TARGET_F1 = 0.70
TARGET_DEPLOY_PRECISION = 0.75
MIN_DEPLOY_RECALL = 0.10
HIGH_PRECISION_SCORE_FLOOR = 0.70
BALANCED_SCORE_FLOOR = 0.25
EXPORT_ONNX = False
EXPORT_TENSORRT = False

PROFILES = {
    "fast_smoke": dict(
        device=0, workers=2, main_cache="ram", refine_cache="ram",
        main_images=400, main_imgsz=512, main_batch=4,
        main_epochs=1, main_hours=None, main_freeze=6,
        refine_images=200, refine_imgsz=576, refine_batch=4,
        refine_epochs=1, refine_hours=None, epoch_val_images=200,
        eval_imgsz=576, eval_batch=4,
    ),
    "fast_overnight": dict(
        device=0, workers=4, main_cache=False, refine_cache="ram",
        main_images=8000, main_imgsz=576, main_batch=10,
        main_epochs=100, main_hours=6.0, main_freeze=6,
        refine_images=1500, refine_imgsz=640, refine_batch=8,
        refine_epochs=60, refine_hours=3.0, epoch_val_images=1200,
        eval_imgsz=640, eval_batch=8,
    ),
    "larger_gpu": dict(
        device=0, workers=8, main_cache="ram", refine_cache="ram",
        main_images=12000, main_imgsz=768, main_batch=0.90,
        main_epochs=80, main_hours=18.0, main_freeze=0,
        refine_images=5000, refine_imgsz=832, refine_batch=0.85,
        refine_epochs=40, refine_hours=8.0, epoch_val_images=3000,
        eval_imgsz=832, eval_batch=0.80,
    ),
    "cpu_fallback": dict(
        device="cpu", workers=0, main_cache=False, refine_cache=False,
        main_images=2000, main_imgsz=512, main_batch=4,
        main_epochs=30, main_hours=6.0, main_freeze=10,
        refine_images=800, refine_imgsz=576, refine_batch=2,
        refine_epochs=15, refine_hours=2.0, epoch_val_images=500,
        eval_imgsz=576, eval_batch=4,
    ),
}

cfg = dict(PROFILES[RUN_MODE])
RUN_ROOT = PROJECT_ROOT / "runs" / "notebooks" / "yolo_accuracy"
DATA_YAML = PROJECT_ROOT / "data" / "bdd100k_yolo" / "data.yaml"
MANIFEST_ROOT = RUN_ROOT / "manifests" / RUN_TAG
MAIN_NAME = f"bdd100k_{RUN_MODE}_{RUN_TAG}_main"
REFINE_NAME = f"bdd100k_{RUN_MODE}_{RUN_TAG}_refine"
MAIN_RUN_DIR = RUN_ROOT / MAIN_NAME
REFINE_RUN_DIR = RUN_ROOT / REFINE_NAME

BASELINE_WEIGHTS = (
    PROJECT_ROOT / "runs" / "notebooks" / "yolo"
    / "bdd100k_cpu_quick_finetuned" / "weights" / "best.pt"
)
if START_WEIGHTS_OVERRIDE:
    START_WEIGHTS = Path(START_WEIGHTS_OVERRIDE)
    if not START_WEIGHTS.is_absolute():
        START_WEIGHTS = PROJECT_ROOT / START_WEIGHTS
else:
    try:
        START_WEIGHTS = latest_best_checkpoint(
            RUN_ROOT,
            fallback=BASELINE_WEIGHTS,
            exclude_run_names=(MAIN_NAME, REFINE_NAME),
        )
    except FileNotFoundError:
        START_WEIGHTS = "yolo11n.pt"
        print("No local BDD100K checkpoint found; starting from COCO YOLO11n.")

print(f"Starting checkpoint: {START_WEIGHTS}")
display(pd.DataFrame([cfg], index=[RUN_MODE]).T.rename(columns={RUN_MODE: "value"}))

## Verify CUDA and power state

For an overnight run, connect the laptop charger and use the Windows Best performance
power mode. CUDA training is stopped immediately if this notebook is accidentally using
a CPU-only PyTorch build.

In [ ]:
if cfg["device"] != "cpu" and not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Restart Jupyter with C:\tf214_hw2 after installing "
        "the CUDA-enabled PyTorch wheel."
    )

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name} | VRAM: {gpu.total_memory / 1024**3:.1f} GB")
    x = torch.randn(1024, 1024, device="cuda")
    _ = x @ x
    torch.cuda.synchronize()
    del x
    torch.cuda.empty_cache()
else:
    print("CPU fallback selected. Accuracy gained per hour will be much lower.")

## Reuse the balanced manifests and create a small epoch-validation split

On this computer all class-aware `v2` images are retained and fresh scenes are added in
seconds. A newly converted
dataset uses `data/bdd100k_yolo/index/train.jsonl`, which the updated converter creates
while labels are already in memory. If neither exists, the fallback uses uniform image
sampling and clearly reports that choice.

In [ ]:
assert DATA_YAML.exists(), f"Missing converted dataset: {DATA_YAML}"
data_config, dataset_root, class_names = load_dataset_config(DATA_YAML)
fast_data = prepare_fast_data_files(
    data_yaml=DATA_YAML,
    output_dir=MANIFEST_ROOT,
    run_root=RUN_ROOT,
    preferred_manifest_tag=PREFERRED_MANIFEST_TAG,
    main_count=cfg["main_images"],
    refine_count=cfg["refine_images"],
    validation_count=cfg["epoch_val_images"],
    seed=SEED,
)
MAIN_DATA_YAML = fast_data.main_yaml
REFINE_DATA_YAML = fast_data.refine_yaml

manifest_summary = pd.DataFrame([
    {"purpose": "main training", "images": sum(1 for _ in fast_data.main_manifest.open(encoding="utf-8"))},
    {"purpose": "rare-class refinement", "images": sum(1 for _ in fast_data.refine_manifest.open(encoding="utf-8"))},
    {"purpose": "per-epoch validation", "images": sum(1 for _ in fast_data.validation_manifest.open(encoding="utf-8"))},
])
print(f"Manifest source: {fast_data.source}")
display(manifest_summary)

## Compare against the measured `v2` run

This diagnostic reads the interrupted run without modifying it. The final `v2` epoch
reached 0.347 mAP@50 and was still improving, which is why this notebook continues its
best checkpoint instead of starting over.

In [ ]:
start_path = Path(START_WEIGHTS)
prior_results = start_path.parent.parent / "results.csv" if start_path.exists() else None
if prior_results is not None and prior_results.exists():
    prior_history = pd.read_csv(prior_results)
    time_column = "time"
    epoch_seconds = prior_history[time_column].diff().fillna(prior_history[time_column])
    prior_display = prior_history[[
        "epoch", "metrics/precision(B)", "metrics/recall(B)",
        "metrics/mAP50(B)", "metrics/mAP50-95(B)"
    ]].copy()
    prior_display["epoch_minutes"] = epoch_seconds / 60.0
    display(prior_display.tail().style.format({
        "metrics/precision(B)": "{:.3f}",
        "metrics/recall(B)": "{:.3f}",
        "metrics/mAP50(B)": "{:.3f}",
        "metrics/mAP50-95(B)": "{:.3f}",
        "epoch_minutes": "{:.1f}",
    }))
else:
    print("No prior results.csv was found; training will still continue from the checkpoint.")

## Training helper

Main training freezes only the earliest feature layers, which reduces backward-pass cost
while preserving the BDD100K features already learned. Refinement unfreezes the full
network at 640 pixels. The 8,000-image main set streams from the local SSD to avoid RAM
pressure; only the 1,500-image refinement set is cached in RAM.

In [ ]:
def train_stage(start_weights, data_yaml, run_name, stage, resume=False):
    run_dir = RUN_ROOT / run_name
    last_weights = run_dir / "weights" / "last.pt"
    if resume:
        assert last_weights.exists(), f"No resumable checkpoint: {last_weights}"
        print(f"Resuming {stage} from {last_weights}")
        return YOLO(str(last_weights)).train(resume=True)
    if run_dir.exists():
        raise FileExistsError(
            f"{run_dir} already exists. Set RESUME_{stage.upper()}=True or change RUN_TAG."
        )

    is_main = stage == "main"
    train_args = dict(
        data=str(data_yaml),
        epochs=cfg["main_epochs"] if is_main else cfg["refine_epochs"],
        imgsz=cfg["main_imgsz"] if is_main else cfg["refine_imgsz"],
        batch=cfg["main_batch"] if is_main else cfg["refine_batch"],
        device=cfg["device"],
        workers=cfg["workers"],
        cache=cfg["main_cache"] if is_main else cfg["refine_cache"],
        project=str(RUN_ROOT),
        name=run_name,
        exist_ok=False,
        pretrained=True,
        optimizer="AdamW",
        lr0=4e-4 if is_main else 2e-4,
        lrf=0.08,
        weight_decay=6e-4,
        patience=12 if is_main else 8,
        cos_lr=True,
        warmup_epochs=0.5,
        box=7.5,
        cls=0.55 if is_main else 0.65,
        cls_pw=0.45 if is_main else 0.65,
        mosaic=0.55 if is_main else 0.25,
        mixup=0.0,
        hsv_h=0.015,
        hsv_s=0.40 if is_main else 0.30,
        hsv_v=0.40,
        translate=0.08 if is_main else 0.05,
        scale=0.45 if is_main else 0.25,
        fliplr=0.50,
        close_mosaic=5 if is_main else 3,
        amp=cfg["device"] != "cpu",
        channels_last=cfg["device"] != "cpu",
        deterministic=False,
        seed=SEED,
        max_det=300,
        plots=False,
        save=True,
        save_period=2,
        val=True,
        verbose=True,
    )
    freeze = cfg["main_freeze"] if is_main else 0
    if freeze:
        train_args["freeze"] = freeze
    hours = cfg["main_hours"] if is_main else cfg["refine_hours"]
    if hours is not None:
        train_args["time"] = hours
    return YOLO(str(start_weights)).train(**train_args)

## Stage 1 - efficient continuation

In [ ]:
if RUN_MAIN_TRAINING:
    main_result = train_stage(
        START_WEIGHTS, MAIN_DATA_YAML, MAIN_NAME, "main", resume=RESUME_MAIN
    )
MAIN_BEST = MAIN_RUN_DIR / "weights" / "best.pt"
assert MAIN_BEST.exists(), f"Main checkpoint not found: {MAIN_BEST}"
MAIN_BEST

## Stage 2 - full-network rare-class refinement

In [ ]:
if RUN_REFINEMENT:
    refine_result = train_stage(
        MAIN_BEST, REFINE_DATA_YAML, REFINE_NAME, "refinement", resume=RESUME_REFINEMENT
    )
REFINE_BEST = REFINE_RUN_DIR / "weights" / "best.pt"
assert REFINE_BEST.exists(), f"Refinement checkpoint not found: {REFINE_BEST}"
REFINE_BEST

## Learning curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for run_dir, label in ((MAIN_RUN_DIR, "main"), (REFINE_RUN_DIR, "refined")):
    results_path = run_dir / "results.csv"
    if not results_path.exists():
        continue
    history = pd.read_csv(results_path)
    axes[0].plot(history["epoch"], history["metrics/mAP50(B)"], marker="o", label=label)
    precision = history["metrics/precision(B)"]
    recall = history["metrics/recall(B)"]
    f1 = 2 * precision * recall / (precision + recall).clip(lower=1e-12)
    axes[1].plot(history["epoch"], f1, marker="o", label=label)
axes[0].set(title="Epoch validation mAP@50", xlabel="epoch", ylabel="mAP@50")
axes[1].set(title="Epoch validation F1", xlabel="epoch", ylabel="F1")
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend()
plt.tight_layout()
plt.show()

## Fast candidate screening, then full validation

All available checkpoints are screened on the same fixed 1,200-image subset. The
starting checkpoint and strongest newly trained checkpoint are then evaluated on all
8,000 validation images. This guarantees a no-regression comparison while avoiding a
third expensive full validation pass.

In [ ]:
def summarize_metrics(label, metrics):
    precision = float(metrics.box.mp)
    recall = float(metrics.box.mr)
    f1 = 2 * precision * recall / max(1e-12, precision + recall)
    map50 = float(metrics.box.map50)
    map50_95 = float(metrics.box.map)
    return {
        "candidate": label,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mAP50": map50,
        "mAP50_95": map50_95,
        "selection_score": 0.60 * f1 + 0.40 * map50,
    }

candidate_paths = {
    "starting": START_WEIGHTS,
    "main": MAIN_BEST,
    "refined": REFINE_BEST,
}
quick_rows = []
for label, weights in candidate_paths.items():
    metrics = YOLO(str(weights)).val(
        data=str(MAIN_DATA_YAML), split="val",
        imgsz=cfg["eval_imgsz"], batch=cfg["eval_batch"],
        device=cfg["device"], workers=cfg["workers"],
        conf=0.001, iou=0.70, max_det=300, plots=False, verbose=False,
    )
    quick_rows.append(summarize_metrics(label, metrics))

quick_table = pd.DataFrame(quick_rows).sort_values("selection_score", ascending=False)
display(quick_table.style.format({
    column: "{:.3f}" for column in quick_table.columns if column != "candidate"
}))
ranked_trained = [
    label for label in quick_table["candidate"].tolist() if label != "starting"
]
finalist_labels = ["starting", ranked_trained[0]]
print(f"Full-validation finalists: {finalist_labels}")

In [ ]:
full_val_metrics = {}
full_rows = []
for label in finalist_labels:
    metrics = YOLO(str(candidate_paths[label])).val(
        data=str(DATA_YAML), split="val",
        imgsz=cfg["eval_imgsz"], batch=cfg["eval_batch"],
        device=cfg["device"], workers=cfg["workers"],
        conf=0.001, iou=0.70, max_det=300, plots=True,
        project=str(RUN_ROOT), name=f"{MAIN_NAME}_{label}_full_val",
        verbose=False,
    )
    full_val_metrics[label] = metrics
    full_rows.append(summarize_metrics(label, metrics))

candidate_table = pd.DataFrame(full_rows).sort_values("selection_score", ascending=False)
display(candidate_table.style.format({
    column: "{:.3f}" for column in candidate_table.columns if column != "candidate"
}))
SELECTED_LABEL = str(candidate_table.iloc[0]["candidate"])
BEST_WEIGHTS = candidate_paths[SELECTED_LABEL]
val_metrics = full_val_metrics[SELECTED_LABEL]
best_model = YOLO(str(BEST_WEIGHTS))
print(f"Selected checkpoint: {SELECTED_LABEL} -> {BEST_WEIGHTS}")

## Held-out test evaluation

In [ ]:
test_metrics = best_model.val(
    data=str(DATA_YAML), split="test",
    imgsz=cfg["eval_imgsz"], batch=cfg["eval_batch"],
    device=cfg["device"], workers=cfg["workers"],
    conf=0.001, iou=0.70, max_det=300, plots=True,
    project=str(RUN_ROOT), name=f"{MAIN_NAME}_{SELECTED_LABEL}_test",
    verbose=False,
)

summary = pd.DataFrame([
    {**summarize_metrics("validation", val_metrics), "split": "validation"},
    {**summarize_metrics("test", test_metrics), "split": "test"},
]).drop(columns=["candidate", "selection_score"])
display(summary.style.format({
    column: "{:.3f}" for column in summary.columns if column != "split"
}))

In [ ]:
def per_class_table(metrics):
    rows = []
    metric_position = {
        int(class_id): position
        for position, class_id in enumerate(metrics.ap_class_index)
    }
    for class_id, name in best_model.names.items():
        position = metric_position.get(class_id)
        precision = float(metrics.box.p[position]) if position is not None else 0.0
        recall = float(metrics.box.r[position]) if position is not None else 0.0
        rows.append({
            "class": name,
            "precision": precision,
            "recall": recall,
            "f1": 2 * precision * recall / max(1e-12, precision + recall),
            "mAP50": float(metrics.box.ap50[position]) if position is not None else 0.0,
            "mAP50_95": float(metrics.box.maps[class_id]),
        })
    return pd.DataFrame(rows)

print("Validation by class")
display(per_class_table(val_metrics).style.format("{:.3f}", subset=[
    "precision", "recall", "f1", "mAP50", "mAP50_95"
]))
print("Test by class")
display(per_class_table(test_metrics).style.format("{:.3f}", subset=[
    "precision", "recall", "f1", "mAP50", "mAP50_95"
]))

## Two confidence profiles

`balanced` maximizes validation F1 and is useful for analysis. `high_precision` targets
75% validation precision and never displays a box below 0.70 confidence. A 0.70 score
floor guarantees that displayed score range, but it does not guarantee 70% mAP and it
can reduce recall. Live detection defaults to the high-precision profile.

In [ ]:
def calibrate_profile(metrics, name, score_floor, precision_target=None):
    px = np.asarray(metrics.box.px)
    f1_curve = np.asarray(metrics.box.f1_curve)
    p_curve = np.asarray(metrics.box.p_curve)
    r_curve = np.asarray(metrics.box.r_curve)
    rows = []
    thresholds = {}

    for position, class_id_raw in enumerate(metrics.ap_class_index):
        class_id = int(class_id_raw)
        best_f1_index = int(np.nanargmax(f1_curve[position]))
        selected_index = best_f1_index
        selection = "best F1"
        if precision_target is not None:
            valid = np.flatnonzero(
                (p_curve[position] >= precision_target)
                & (r_curve[position] >= MIN_DEPLOY_RECALL)
            )
            if len(valid):
                selected_index = int(valid[np.nanargmax(f1_curve[position, valid])])
                selection = "precision target"
            else:
                selection = "best F1 fallback"

        threshold = max(score_floor, float(px[selected_index]))
        thresholds[str(class_id)] = threshold
        rows.append({
            "profile": name,
            "class_id": class_id,
            "class": best_model.names[class_id],
            "threshold": threshold,
            "estimated_precision": float(np.interp(threshold, px, p_curve[position])),
            "estimated_recall": float(np.interp(threshold, px, r_curve[position])),
            "best_f1": float(f1_curve[position, best_f1_index]),
            "selection": selection,
        })
    return thresholds, pd.DataFrame(rows)

balanced_thresholds, balanced_calibration = calibrate_profile(
    val_metrics, "balanced", BALANCED_SCORE_FLOOR
)
high_precision_thresholds, high_precision_calibration = calibrate_profile(
    val_metrics, "high_precision", HIGH_PRECISION_SCORE_FLOOR,
    precision_target=TARGET_DEPLOY_PRECISION,
)
calibration_table = pd.concat(
    [balanced_calibration, high_precision_calibration], ignore_index=True
)
display(calibration_table.style.format({
    "threshold": "{:.3f}",
    "estimated_precision": "{:.3f}",
    "estimated_recall": "{:.3f}",
    "best_f1": "{:.3f}",
}))

## Save the deployment model and calibrated configuration

In [ ]:
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "yolo_accuracy_fast"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DEPLOY_WEIGHTS = OUTPUT_ROOT / "bdd100k_yolo_fast_best.pt"
DEPLOY_CONFIG = OUTPUT_ROOT / "deployment_config.json"
shutil.copy2(BEST_WEIGHTS, DEPLOY_WEIGHTS)

threshold_profiles = {
    "balanced": {
        "inference_confidence": min(balanced_thresholds.values()),
        "class_thresholds": balanced_thresholds,
    },
    "high_precision": {
        "inference_confidence": min(high_precision_thresholds.values()),
        "class_thresholds": high_precision_thresholds,
    },
}
deployment = {
    "model_family": "YOLO11",
    "selected_stage": SELECTED_LABEL,
    "weights": str(DEPLOY_WEIGHTS.resolve()),
    "imgsz": cfg["eval_imgsz"],
    "max_det": 100,
    "default_threshold_profile": "high_precision",
    "threshold_profiles": threshold_profiles,
    "inference_confidence": threshold_profiles["high_precision"]["inference_confidence"],
    "class_thresholds": high_precision_thresholds,
    "class_names": {str(key): value for key, value in best_model.names.items()},
    "target_validation_precision": TARGET_DEPLOY_PRECISION,
    "minimum_display_confidence": HIGH_PRECISION_SCORE_FLOOR,
    "validation_metrics": summarize_metrics("validation", val_metrics),
    "test_metrics": summarize_metrics("test", test_metrics),
}
DEPLOY_CONFIG.write_text(json.dumps(deployment, indent=2), encoding="utf-8")
print(f"Weights: {DEPLOY_WEIGHTS}")
print(f"Configuration: {DEPLOY_CONFIG}")
print(
    "Live command:\n"
    f'python -m road_detection.realtime_detect --backend yolo '
    f'--weights "{DEPLOY_WEIGHTS}" --config "{DEPLOY_CONFIG}" '
    f'--threshold-profile high_precision --source 0 --device 0'
)

## Calibrated predictions

In [ ]:
deployment_model = YOLO(str(DEPLOY_WEIGHTS))
class_thresholds = high_precision_thresholds
inference_confidence = min(class_thresholds.values())

def calibrated_frame(result):
    frame = result.orig_img.copy()
    coordinates = result.boxes.xyxy.detach().cpu().numpy()
    class_ids = result.boxes.cls.detach().cpu().numpy().astype(np.int32)
    scores = result.boxes.conf.detach().cpu().numpy()
    for box, class_id, score in zip(coordinates, class_ids, scores):
        if score < class_thresholds.get(str(int(class_id)), HIGH_PRECISION_SCORE_FLOOR):
            continue
        x1, y1, x2, y2 = [int(value) for value in box]
        color = ((37 * int(class_id) + 60) % 255, (91 * int(class_id) + 110) % 255, (151 * int(class_id) + 80) % 255)
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        text = f"{deployment_model.names[int(class_id)]} {score:.2f}"
        cv2.putText(frame, text, (x1, max(18, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

test_dir = dataset_root / "images" / "test"
test_images = sorted(test_dir.glob("*.jpg"))
sample_images = random.Random(SEED).sample(test_images, min(6, len(test_images)))
predictions = deployment_model.predict(
    sample_images, imgsz=cfg["eval_imgsz"], conf=inference_confidence,
    device=cfg["device"], quantize=16 if cfg["device"] != "cpu" else None,
    max_det=100, verbose=False,
)
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for axis, result, image_path in zip(axes.flat, predictions, sample_images):
    axis.imshow(calibrated_frame(result))
    axis.set_title(image_path.name)
    axis.axis("off")
for axis in axes.flat[len(predictions):]:
    axis.axis("off")
plt.tight_layout()
plt.show()

## FP16 real-time benchmark

In [ ]:
benchmark_images = test_images[:min(40, len(test_images))]
benchmark_frames = [cv2.imread(str(path)) for path in benchmark_images]
benchmark_frames = [frame for frame in benchmark_frames if frame is not None]
deployment_model.fuse()
for frame in benchmark_frames[:5]:
    deployment_model.predict(
        frame, imgsz=cfg["eval_imgsz"], conf=inference_confidence,
        device=cfg["device"], quantize=16 if cfg["device"] != "cpu" else None,
        max_det=100, verbose=False,
    )
if torch.cuda.is_available():
    torch.cuda.synchronize()

inference_times = []
started = time.perf_counter()
for frame in benchmark_frames:
    result = deployment_model.predict(
        frame, imgsz=cfg["eval_imgsz"], conf=inference_confidence,
        device=cfg["device"], quantize=16 if cfg["device"] != "cpu" else None,
        max_det=100, verbose=False,
    )[0]
    inference_times.append(float(result.speed["inference"]))
if torch.cuda.is_available():
    torch.cuda.synchronize()
elapsed = time.perf_counter() - started
end_to_end_fps = len(benchmark_frames) / max(elapsed, 1e-9)
inference_fps = 1000.0 / max(float(np.mean(inference_times)), 1e-9)
print(f"Camera-like end-to-end speed: {end_to_end_fps:.2f} FPS")
print(f"Neural-network inference only: {inference_fps:.2f} FPS")

report = {
    **deployment,
    "end_to_end_fps": end_to_end_fps,
    "model_inference_fps": inference_fps,
    "profile": RUN_MODE,
    "run_tag": RUN_TAG,
    "candidate_comparison": candidate_table.to_dict(orient="records"),
}
(OUTPUT_ROOT / "final_evaluation.json").write_text(
    json.dumps(report, indent=2), encoding="utf-8"
)
summary.to_csv(OUTPUT_ROOT / "final_metrics.csv", index=False)
calibration_table.to_csv(OUTPUT_ROOT / "confidence_calibration.csv", index=False)

## Optional optimized exports

The `.pt` file already uses FP16 CUDA inference in the live command. TensorRT can be
faster, but export time and dependency installation are intentionally opt-in.

In [ ]:
exported = {}
if EXPORT_ONNX:
    exported["onnx"] = deployment_model.export(
        format="onnx", imgsz=cfg["eval_imgsz"], dynamic=False,
        simplify=True, half=False
    )
if EXPORT_TENSORRT:
    if not torch.cuda.is_available():
        raise RuntimeError("TensorRT export requires CUDA.")
    exported["engine"] = deployment_model.export(
        format="engine", imgsz=cfg["eval_imgsz"], batch=1,
        dynamic=False, simplify=True, half=True, device=0
    )
exported

## Acceptance decision

Validation is used for model selection and confidence calibration. The held-out test
split is reported once. If the measured target remains unmet, the valid next step is a
larger model or more GPU time; raising a display threshold only changes the precision
versus recall operating point.

In [ ]:
acceptance = summary[["split", "precision", "recall", "f1", "mAP50"]].copy()
acceptance["target_f1"] = TARGET_F1
acceptance["passed_f1"] = acceptance["f1"] >= TARGET_F1
display(acceptance.style.format({
    "precision": "{:.3f}", "recall": "{:.3f}", "f1": "{:.3f}",
    "mAP50": "{:.3f}", "target_f1": "{:.2f}",
}))
if acceptance["passed_f1"].all():
    print("The measured validation/test F1 target is met.")
else:
    print(
        "The measured F1 target is not yet met. The saved high-precision profile "
        "still displays only detections with confidence >= 0.70."
    )